# NICAR Lightning Talks Data Processing

This notebook processes NICAR lightning talk data from 2010-2025:
1. Loads and cleans CSV data
2. Extracts themes using keyword matching
3. Calculates aggregations for visualizations
4. Exports processed JSON file

In [70]:
import pandas as pd
import numpy as np
import json
import re
from collections import Counter, defaultdict

## 1. Load and Clean Data

In [74]:
# Load CSV
df = pd.read_csv('lightningtalks_clean.csv')

df.head(20)

,year,title,speaker,org,M/F,nonmale,location,international,description,theme
0,2010,"Hello, Newsroom! Build out a GIS-enabled web a...",Brian Boyer,Chicago Tribune,M,0,"Chicago, IL",0.0,Fire up a server in Amazon's cloud and deploy ...,GIS
1,2010,"Google Charts. Easy, Clear, Indestructable.",Ben Welsh,Los Angeles Times,M,0,"Los Angeles, CA",0.0,They aren't interactive. They don't impress tr...,Charts
2,2010,Data Manipulation or Graphics with R,Bill Alpert,Barron's,M,0,"New York, NY",0.0,I could show how handily R allows you to munge...,R
3,2010,Using an API with Excel,Derek Willis,New York Times,M,0,"Silver Spring, MD",0.0,Want to use an API to fetch data but don't hav...,APIs
4,2010,"Like Snowboard Cross, but With Data",Scott Klein,ProPublica,M,0,"New York, NY",0.0,Use ProPublica's new open source tools (to be ...,"OSINT, Proprietary Tools"
5,2010,Essential Queries for SQL Server,Anthony DeBarros,USA Today,M,0,"McLean, VA",0.0,Five SQL queries you may have never tried that...,Viz Tools
6,2010,The new free visualization tool,Sarah Cohen,Reporter's Lab,F,1,"Durham, NC",0.0,Tableau public takes on ManyEyes,SQL
7,2010,The hidden power of Javascript,Michelle Minkoff,Medill School of Journalism,F,1,"Washington, DC",0.0,"If you know some HTML and CSS, Javascript is a...",Viz Tools
8,2010,Easy peasy due diligence,Danielle Cervantes,San Diego Union-Tribune,F,1,"San Diego, CA",0.0,Roll through a seven-step process to vet sourc...,Data cleaning
9,2010,Easy interactive charts with Open Flash Charts,Tim Henderson,Pew Charitable Trusts,M,0,"Washington, DC",0.0,Open Flash Charts is an easy way to make inter...,Charts


In [ ]:
#Talks per year
print("Talks per year:")
print(df['year'].value_counts().sort_index())

Talks per year:
year
2010    10
2011    10
2012    10
2013    10
2014    10
2015    10
2016    10
2017    10
2018    10
2019    10
2020    10
2021    11
2022    10
2023    10
2024    10
2025    10
Name: count, dtype: int64


In [107]:
# Clean and exract the gender field

gender_counts = df.groupby(['year', 'M/F']).size().reset_index(name='count')                                                                                                                                                                                                         
print(gender_counts)


    year M/F  count
0   2010   F      3
1   2010   M      7
2   2011   F      1
3   2011   M      9
4   2012   M     10
5   2013   F      2
6   2013   M      8
7   2014   F      4
8   2014   M      6
9   2015   F      5
10  2015   M      5
11  2016   F      4
12  2016   M      6
13  2017   F      5
14  2017   M      5
15  2018   F      4
16  2018   M      6
17  2019   F      5
18  2019   M      5
19  2020   F      5
20  2020   M      5
21  2021   F      8
22  2021   M      3
23  2022   F      4
24  2022   M      6
25  2023   F      2
26  2023   M      8
27  2024   F      4
28  2024   M      6
29  2025   F      7
30  2025   M      3


## 3. Extract Themes Using Keyword Matching

In [102]:
# Define theme keywords (case-insensitive matching)
theme_keywords = {
    'Excel': ['excel', 'spreadsheet', 'pivot', 'vlookup', 'formula'],
    'Python': ['python', 'django', 'flask', 'pandas', 'numpy'],
    'JavaScript': ['javascript', ' js ', 'jquery', 'node', 'react', 'angular', 'vue'],
    'R': [' r ', 'rstats', 'ggplot', 'tidyverse'],
    'Scraping': ['scrap', 'crawl', 'beautifulsoup', 'selenium'],
    'APIs': [' api', 'api '],
    'Elections': ['election', 'vote', 'voting', 'ballot', 'polling'],
    'Databases': ['database', ' sql', 'mysql', 'postgresql', 'mongodb','soccer','sports','sqlite'],
    'Mapping/GIS': ['map', ' gis', 'geodjango', 'geospatial', 'geocod', 'cartography', 'leaflet', 'mapbox'],
    'Visualization': ['visualiz', 'chart', 'graph', 'infographic', ' d3', 'tableau', 'datawrapper'],
    'AI/ML/NLP': ['ai','NLP','machine learning', 'gpt', 'chatgpt', 'algorithm', 'neural', 'openai', 'llm'],
    'Cloud/Infrastructure': ['aws', 'cloud', 'server', 'deploy', 'heroku', 'docker', 'kubernetes'],
    'FOIA/Records': ['foia', 'public record', 'archive', 'wayback','fees', 'freedom of information'],
    'Security/Privacy': ['security', 'encrypt', 'privacy', 'password', 'hack'],
    'Satire/Humor': ['cat','5','five','kitten', 'mosquito', 'food', 'humor', 'funny', 'satire', 'bunny', 'pet', 'joke'],
    'Tools/Workflow': ['workflow', 'productivity', 'git', 'github', 'version control'],
    'Data Cleaning': ['clean', 'munge', 'wrangle', 'normalize', 'standardize'],
    'Statistics': ['statistic', 'regression', 'correlation', 'probability', 'hypothesis'],
    'Mobile': ['mobile', 'iphone', 'android', 'app ', 'smartphone'],
    'Data Habits':['archive', 'habits', 'workflow', 'git', 'productivity', 'multi-task', 'version control'],
    'Storytime':['passion','enemy','old','past','daughter', 'growing up', 'story', 'personal', 'life', 'experience', 'journey', 'career'],
}

def extract_themes(title, description):
    """Extract themes from title and description using keyword matching."""
    # Combine title and description for searching
    text = f"{title} {description}".lower()
    
    matched_themes = []
    
    for theme, keywords in theme_keywords.items():
        for keyword in keywords:
            if keyword.lower() in text:
                matched_themes.append(theme)
                break  # Only add theme once even if multiple keywords match
    
    # If no themes found, assign "Other"
    if not matched_themes:
        matched_themes = ['Other']
    
    return matched_themes

In [103]:
# Ensure df_filtered exists (in case cells were run out of order)
if 'df_filtered' not in globals():
    # Recreate filtering logic from earlier cell so this cell is safe to run independently
    early_years = df[df['year'] <= 2018].copy()
    late_years = df[df['year'] >= 2019].copy()

    # top 10 per year were fixed manually — just use the early_years dataframe as-is
    top_10_early = early_years.copy()

    df_filtered = pd.concat([top_10_early, late_years], ignore_index=True)
    # Keep existing order within each year (ignore votes) and reset the index
    df_filtered = df_filtered.sort_values(['year'], ascending=[True]).reset_index(drop=True)

# Apply theme extraction
df_filtered['themes'] = df_filtered.apply(
    lambda row: extract_themes(str(row['title']), str(row['description'])),
    axis=1
)

# Show some examples (safe against small datasets)
print("\nExample talks with extracted themes:")
for idx in [0, 10, 50, 100]:
    if idx < len(df_filtered):
        row = df_filtered.iloc[idx]
        print(f"\n{row['year']}: {row['title'][:60]}...")
        print(f"Themes: {', '.join(row['themes'])}")
    else:
        print(f"\nIndex {idx} out of range (len={len(df_filtered)})")


Example talks with extracted themes:

2010: Hello, Newsroom! Build out a GIS-enabled web app in < five m...
Themes: Mapping/GIS, Cloud/Infrastructure, Satire/Humor, Mobile

2011: Similarity algorithms...
Themes: Python, Elections, AI/ML/NLP

2014: Practical Calculus...
Themes: Other

2019: Save Student Newsrooms. How you can help the next generation...
Themes: Satire/Humor


In [104]:
# Count theme frequency
all_themes = []
for themes in df_filtered['themes']:
    all_themes.extend(themes)

theme_counts = Counter(all_themes)
print("\nTheme frequency across all talks:")
for theme, count in theme_counts.most_common():
    print(f"{theme}: {count}")


Theme frequency across all talks:
Satire/Humor: 54
Storytime: 48
AI/ML/NLP: 47
Other: 24
Visualization: 22
Mapping/GIS: 15
Python: 14
Databases: 12
Excel: 11
Data Habits: 11
FOIA/Records: 10
Cloud/Infrastructure: 9
Mobile: 9
JavaScript: 8
Tools/Workflow: 8
Elections: 7
Data Cleaning: 6
APIs: 6
Security/Privacy: 5
Scraping: 4
R: 2
Statistics: 1


## 4. Calculate Aggregations for Visualizations

In [108]:
# Gender counts by year
gender_counts = df.groupby(['year', 'M/F']).size().unstack(fill_value=  0)                                                                                                                                                                                                                                                                                                                                           
print(gender_counts)

M/F   F   M
year       
2010  3   7
2011  1   9
2012  0  10
2013  2   8
2014  4   6
2015  5   5
2016  4   6
2017  5   5
2018  4   6
2019  5   5
2020  5   5
2021  8   3
2022  4   6
2023  2   8
2024  4   6
2025  7   3


In [60]:
# Theme counts by year
themes_by_year = defaultdict(lambda: defaultdict(int))

for _, row in df_filtered.iterrows():
    year = row['year']
    for theme in row['themes']:
        themes_by_year[year][theme] += 1

# Show sample
print("\nThemes in 2015:")
for theme, count in sorted(themes_by_year[2015].items(), key=lambda x: x[1], reverse=True):
    print(f"  {theme}: {count}")




Themes in 2015:
  Other: 5
  Visualization: 3
  Mapping/GIS: 2
  Cloud/Infrastructure: 1
  Excel: 1
  JavaScript: 1
  R: 1
  Python: 1
  Databases: 1
  Satire/Humor: 1


In [61]:
# Theme trends over time (for line graphs)
years = sorted(df_filtered['year'].unique())
theme_trends = []

for theme in sorted(theme_counts.keys()):
    if theme == 'Other':  # Skip 'Other' category for trends
        continue
    
    trend_data = []
    for year in years:
        count = themes_by_year[year].get(theme, 0)
        trend_data.append({'year': int(year), 'count': int(count)})
    
    theme_trends.append({
        'theme': theme,
        'data': trend_data
    })

# Show one example
print("\nExample trend - Satire/Humor over years:")
python_trend = next((t for t in theme_trends if t['theme'] == 'Satire/Humor'), None)
if python_trend:
    for item in python_trend['data']:
        print(f"  {item['year']}: {item['count']}")


Example trend - Satire/Humor over years:
  2010: 0
  2011: 0
  2012: 2
  2013: 3
  2014: 2
  2015: 1
  2016: 2
  2017: 0
  2018: 1
  2019: 2
  2020: 0
  2021: 1
  2022: 1
  2023: 2
  2024: 1
  2025: 2


## 5. Export JSON for Website

In [62]:
# Create data directory if it doesn't exist
import os
os.makedirs('data', exist_ok=True)

In [63]:
# Prepare talks data
talks = []
for idx, row in df_filtered.iterrows():
    talk = {
        'id': idx + 1,
        'year': int(row['year']),
        'title': str(row['title']),
        'speaker': str(row['speaker']),
        'org': str(row['org']) if pd.notna(row['org']) else '',
        'gender': str(row['gender']),
        'themes': row['themes'],
        'description': str(row['description'])[:500]  # Truncate long descriptions
    }
    talks.append(talk)

print(f"\nPrepared {len(talks)} talks for export")


Prepared 162 talks for export


In [64]:
# Prepare aggregations
gender_by_year_dict = {}
for year in years:
    gender_by_year_dict[int(year)] = {
        'Male': int(gender_by_year.loc[year, 'Male']) if 'Male' in gender_by_year.columns else 0,
        'Female': int(gender_by_year.loc[year, 'Female']) if 'Female' in gender_by_year.columns else 0,
        'Other': int(gender_by_year.loc[year, 'Other']) if 'Other' in gender_by_year.columns else 0
    }

themes_by_year_dict = {}
for year in years:
    themes_by_year_dict[int(year)] = {k: int(v) for k, v in themes_by_year[year].items()}

In [111]:
# Create final JSON structure
output_data = {
    'talks': talks,
    'aggregations': {
        'themesByYear': themes_by_year_dict,
        'themeTrends': theme_trends
    },
    'metadata': {
        'totalTalks': len(talks),
        'years': [int(y) for y in years],
        'themes': sorted([t for t in theme_counts.keys() if t != 'Other'])
    }
}

# Export to JSON
with open('data/talks.json', 'w', encoding='utf-8') as f:
    json.dump(output_data, f, indent=2, ensure_ascii=False)

